# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It works by searching massive document stores for relevant information and using this information to generate answers. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

## Install Dependencies

In [1]:
# Do NOT reinstall torch or torchvision — use Colab's pre-installed versions
!pip install -q datasets pinecone sentence-transformers transformers tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 10.8 MB/s eta 0:00:00


## Load and Prepare Dataset

We use the `wikimedia/wikipedia` dataset in streaming mode. We collect 10,000 passages from articles whose title contains 'history'.

In [2]:
from datasets import load_dataset
from tqdm.auto import tqdm
import pandas as pd

# load dataset in streaming mode
wiki_data = load_dataset(
    'wikimedia/wikipedia',
    '20231101.en',
    split='train',
    streaming=True
).shuffle(seed=960, buffer_size=10_000)

# show available fields
sample = next(iter(wiki_data))
print('Fields:', list(sample.keys()))
print('Title:', sample['title'])
print('Text preview:', sample['text'][:300])


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Fields: ['id', 'url', 'title', 'text']
Title: 57th United States Congress
Text preview: The 57th United States Congress was a meeting of the legislative branch of the United States federal government, composed of the United States Senate and the United States House of Representatives. It met in Washington, DC from March 4, 1901, to March 4, 1903, during the final six months of William 


In [3]:
# filter only documents whose title contains 'history'
history = wiki_data.filter(lambda d: 'history' in d['title'].lower())


In [4]:
# collect 10,000 passages by chunking article text into 500-char pieces
total_doc_count = 10000
counter = 0
docs = []

for d in tqdm(history, total=total_doc_count):
    text = d['text']
    chunks = [text[i:i+500] for i in range(0, min(len(text), 2000), 500)]
    for chunk in chunks:
        if chunk.strip():
            docs.append({
                'article_title': d['title'],
                'section_title': 'History',
                'passage_text':  chunk
            })
    counter += 1
    if counter >= total_doc_count or len(docs) >= total_doc_count:
        break

print(f'Total passages collected: {len(docs)}')


  0%|          | 0/10000 [00:00<?, ?it/s]

Total passages collected: 10003


In [5]:
df = pd.DataFrame(docs)
df.head()


,article_title,section_title,passage_text
0,We Are History,History,We Are History is a British comedy series broa...
1,We Are History,History,"an IKEA store. In another, he recreated the Sp..."
2,We Are History,History,rench visitors who overstayed their welcome an...
3,We Are History,History,"ada!""\nOriginal Air Date: 14 April 2000\n\nSea..."
4,Sam Noble Oklahoma Museum of Natural History,History,The Sam Noble Oklahoma Museum of Natural Histo...


## Initialize Pinecone Index

Get a free API key at [app.pinecone.io](https://app.pinecone.io) and add it to Colab Secrets (🔑 icon in the left sidebar) as `PINECONE_API_KEY`.

In [6]:
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec
import time

pinecone_api_key = userdata.get('PINECONE_API_KEY')
pc = Pinecone(api_key=pinecone_api_key)

# free plan only supports us-east-1
spec = ServerlessSpec(cloud='aws', region='us-east-1')
index_name = 'abstractive-question-answering'

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,
        metric='cosine',
        spec=spec
    )
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

index = pc.Index(index_name)
index.describe_index_stats()


Index 'abstractive-question-answering' already exists.


DescribeIndexStatsResponse(dimension=768, total_vector_count=0, metric='cosine', namespaces=0)

## Initialize Retriever

We use `flax-sentence-embeddings/all_datasets_v3_mpnet-base` — a SentenceTransformer model that outputs 768-dimensional vectors optimized for cosine similarity.

In [7]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

retriever = SentenceTransformer(
    'flax-sentence-embeddings/all_datasets_v3_mpnet-base',
    device=device
)
retriever


Using: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

## Generate Embeddings and Upsert

Encode all passages in batches of 64 and upload to Pinecone with metadata.

In [8]:
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    i_end = min(i + batch_size, len(df))
    batch = df.iloc[i:i_end]
    # generate embeddings for passage_text
    emb = retriever.encode(batch['passage_text'].tolist()).tolist()
    # build metadata
    meta = batch[['article_title', 'section_title', 'passage_text']].to_dict(orient='records')
    # unique IDs
    ids = [str(x) for x in range(i, i_end)]
    index.upsert(vectors=list(zip(ids, emb, meta)))

index.describe_index_stats()


  0%|          | 0/157 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=768, total_vector_count=10003, metric='cosine', namespaces=1)

## Initialize Generator

We use `vblagoje/bart_lfqa` (ELI5 BART). Input format:

>`question: What is X? context: <P> passage one <P> passage two`

In [9]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)
print('Generator ready.')


tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Generator ready.


## Helper Functions

In [10]:
def query_pinecone(query, top_k):
    # encode query and search Pinecone for top_k relevant passages
    xq = retriever.encode([query]).tolist()[0]
    return index.query(vector=xq, top_k=top_k, include_metadata=True)

def format_query(query, context):
    # add <P> tag to each passage and join
    context = ' '.join([f"<P> {m['metadata']['passage_text']}" for m in context])
    return f'question: {query} context: {context}'

def generate_answer(query):
    inputs = tokenizer([query], max_length=1024, return_tensors='pt').to(device)
    ids = generator.generate(inputs['input_ids'], num_beams=2, min_length=20, max_length=40)
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    print(answer)
    return answer


## Run Queries

In [11]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
formatted = format_query(query, result['matches'])
print('Formatted input:')
print(formatted[:500])


Formatted input:
question: when was the first electric power system built? context: <P> 

The first practical use of electricity was for lighting in the Government Printing Office in George Street in April 1883. In 1886, the Roma Street Railway Yards were using arc lights, and in the same year, an underground cable connected the Parliament House from the Printing Office, the first of any Parliament House in Australia. The supervision of the laying of cable was done by E.C. Barton, who formed a company with C.F. 


In [12]:
generate_answer(formatted)


The first electric power system was built in the United States in 1883. The first practical use of electricity was for lighting in the Government Printing Office in George Street in April 1883. In 1886


'The first electric power system was built in the United States in 1883. The first practical use of electricity was for lighting in the Government Printing Office in George Street in April 1883. In 1886'

In [13]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
generate_answer(format_query(query, context['matches']))


The first wireless message was sent by a telegraph. The first telegraph was built in London in 1837. The first telegraph was built in 1839. The first telegraph was built


'The first wireless message was sent by a telegraph. The first telegraph was built in London in 1837. The first telegraph was built in 1839. The first telegraph was built'

In [14]:
# check which passages were used
for doc in context['matches']:
    print(doc['metadata']['passage_text'], end='\n---\n')


ot come into common use until several years later) was sketchy, with magazines such as the November, 1901 issue of Amateur Work showing how to build a simple system based on Hertz' early experiments. Magazines show a continued progress by amateurs including a 1904 story on two Boston, Massachusetts 8th graders constructing a transmitter and receiver with a range of eight miles and a 1906 story about two Rhode Island teenagers building a wireless station in a chicken coop. In the US the first com
---
ame into being after radio waves (proved to exist by Heinrich Rudolf Hertz in 1888) were adapted into a communication system in the 1890s by the Italian inventor Guglielmo Marconi. In the late 19th century there had been amateur wired telegraphers setting up their own interconnected telegraphic systems. Following Marconi's success many people began experimenting with this new form of "wireless telegraphy". Information on "Hertzian wave" based wireless telegraphy systems (the name "radio" wo

In [15]:
# COVID-19 not in history dataset — generator will struggle
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
generate_answer(format_query(query, context['matches']))


COVID-19 is a virus that infects humans. It is believed to have originated in China, where it is believed to have spread to the rest of the world.


'COVID-19 is a virus that infects humans. It is believed to have originated in China, where it is believed to have spread to the rest of the world.'

In [16]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
generate_answer(format_query(query, context['matches']))


The war of currents is a term used to refer to a series of events that occurred in the early 20th century. The most famous of these events was the Battle of Tsushima, which was


'The war of currents is a term used to refer to a series of events that occurred in the early 20th century. The most famous of these events was the Battle of Tsushima, which was'

In [17]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
generate_answer(format_query(query, context['matches']))


The first person to go to the moon was probably Giovanni Cassini, who landed on the moon in 1969. He landed on the moon in 1969.


'The first person to go to the moon was probably Giovanni Cassini, who landed on the moon in 1969. He landed on the moon in 1969.'

In [18]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
generate_answer(format_query(query, context['matches']))


I'm not sure if this counts as a project, but I think it's worth noting that the National Science Foundation's budget for the year was $2.5 billion.


"I'm not sure if this counts as a project, but I think it's worth noting that the National Science Foundation's budget for the year was $2.5 billion."

As we can see, the model can generate some decent answers.

#### Add a few more questions

In [19]:
query = "Who invented television?"
context = query_pinecone(query, top_k=10)
generate_answer(format_query(query, context['matches']))


I'm not sure if this is the right subreddit to ask this question, but I'll give it a shot anyway. I'm not a historian, but I'm a computer science major, so


"I'm not sure if this is the right subreddit to ask this question, but I'll give it a shot anyway. I'm not a historian, but I'm a computer science major, so"

In [20]:
query = "Who designed the Sydney Opera House?"
context = query_pinecone(query, top_k=10)
generate_answer(format_query(query, context['matches']))


The Sydney Opera House was designed by the Irish firm Ireland & Associates. The building was built in the 1960s, and was completed in the 1980s. The building was designed by W. Bur


'The Sydney Opera House was designed by the Irish firm Ireland & Associates. The building was built in the 1960s, and was completed in the 1980s. The building was designed by W. Bur'

In [21]:
query = "How is yoghurt produced?"
context = query_pinecone(query, top_k=10)
generate_answer(format_query(query, context['matches']))


Yogurt is made by adding milk to water and letting it sit for a few days. The milk is then added to the water and the yoghurt is made.


'Yogurt is made by adding milk to water and letting it sit for a few days. The milk is then added to the water and the yoghurt is made.'

## Cleanup

In [22]:
pc.delete_index(index_name)
print(f"Index '{index_name}' deleted.")


Index 'abstractive-question-answering' deleted.
